# Scalable MCP tooling — interactive PoC

Compare the three **tool-exposure strategies** for the Evo MCP server and *observe exactly what the model sees*:

| Strategy | `list_tools()` returns | Mechanism |
|---|---|---|
| `none` | the full catalog (15 demo tools) | today's behavior |
| `tool-search` | `search_tools`, `call_tool` | FastMCP Tool Search transform |
| `code-mode` | `search`, `get_schema`, `execute` | FastMCP Code Mode transform (sandbox) |

This notebook uses a **self-contained demo server** (`demo_server.py`) with mock Evo-flavored
tools that return canned data — **no Evo credentials or network needed**. The same
`apply_strategy()` factory used here is wired into the real server (`src/mcp_tools.py`) behind
the `MCP_TOOL_STRATEGY` env var, so what you see here is the real production wiring.

> Run with: `uv run --extra poc jupyter lab` from the repo root.


## 0. Setup
We talk to the in-memory server with a FastMCP `Client` — the same interface a real MCP client/LLM uses. `list_tools()` is *literally* what the model receives in its context.

In [2]:
import json

from demo_server import build_demo_server
from fastmcp import Client


def show(obj, n=2000):
    s = json.dumps(obj, indent=2, default=str)
    print(s if len(s) <= n else s[:n] + f"\n... [{len(s) - n} more chars]")


async def list_visible(strategy, **kw):
    mcp = build_demo_server(strategy, **kw)
    async with Client(mcp) as c:
        tools = await c.list_tools()
        return [t.name for t in tools]

## 1. What the model sees upfront (`list_tools`)
This is the token cost paid **before the user's request is even read**. Watch it collapse from 15 tools to 2–3 synthetic tools.

In [3]:
for strategy in ["none", "tool-search", "code-mode"]:
    names = await list_visible(strategy)
    print(f"{strategy:12} -> {len(names)} tools: {names}")

none         -> 15 tools: ['list_workspaces', 'get_workspace', 'list_objects', 'get_object', 'get_object_versions', 'count_points', 'list_users', 'workspace_health_check', 'create_workspace', 'create_pointset', 'upload_file', 'kriging_run', 'delete_object', 'remove_user', 'update_user_role']
tool-search  -> 2 tools: ['search_tools', 'call_tool']
code-mode    -> 3 tools: ['search', 'get_schema', 'execute']


### 1a. Approximate context cost of the upfront catalog
The real cost isn't the tool *count* — it's the serialized schema (names + descriptions + params) shipped every turn. Let's measure the full listing for each strategy.

In [4]:
async def listing_size(strategy, **kw):
    mcp = build_demo_server(strategy, **kw)
    async with Client(mcp) as c:
        tools = await c.list_tools()
        payload = [t.model_dump() for t in tools]
    return len(tools), len(json.dumps(payload, default=str))


for strategy in ["none", "tool-search", "code-mode"]:
    count, chars = await listing_size(strategy)
    print(f"{strategy:12} -> {count:2} tools, ~{chars:5} chars of schema (~{chars // 4} tokens) listed upfront")

none         -> 15 tools, ~ 7257 chars of schema (~1814 tokens) listed upfront
tool-search  ->  2 tools, ~ 1316 chars of schema (~329 tokens) listed upfront
code-mode    ->  3 tools, ~ 2527 chars of schema (~631 tokens) listed upfront


## 2. Strategy `none` — the baseline
Every tool is visible; the model calls one, gets the raw result back into context, reasons, calls the next. Simple and deterministic, but the catalog and every intermediate result flow through the context window.

In [5]:
mcp = build_demo_server("none")
async with Client(mcp) as c:
    tools = await c.list_tools()
    print("Visible tools:", [t.name for t in tools])
    print()
    # A 2-step workflow = 2 separate round-trips, 2 raw results in context.
    ws = (await c.call_tool("list_workspaces", {})).data
    print("Step 1 list_workspaces ->")
    show(ws)
    objs = (await c.call_tool("list_objects", {"workspace_id": "ws-001"})).data
    print("Step 2 list_objects(ws-001) ->")
    show(objs)

Visible tools: ['list_workspaces', 'get_workspace', 'list_objects', 'get_object', 'get_object_versions', 'count_points', 'list_users', 'workspace_health_check', 'create_workspace', 'create_pointset', 'upload_file', 'kriging_run', 'delete_object', 'remove_user', 'update_user_role']

Step 1 list_workspaces ->
[
  {
    "id": "ws-001",
    "name": "Copper Ridge",
    "objects": 42
  },
  {
    "id": "ws-002",
    "name": "Gold Valley",
    "objects": 17
  },
  {
    "id": "ws-003",
    "name": "Iron Basin",
    "objects": 88
  }
]
Step 2 list_objects(ws-001) ->
[
  {
    "id": "obj-a",
    "name": "collars",
    "schema": "downhole-collection",
    "size": 1200
  },
  {
    "id": "obj-b",
    "name": "topography",
    "schema": "pointset",
    "size": 98000
  },
  {
    "id": "obj-c",
    "name": "cu_grade",
    "schema": "pointset",
    "size": 45000
  }
]


## 3. Strategy `tool-search` — discovery on demand
`list_tools()` returns only `search_tools` + `call_tool`. The model **searches** for what it needs
(full schema returned inline, ranked by BM25), then invokes via the `call_tool` proxy.

Solves: *too many tools upfront*. Does **not** solve round-trips or intermediate-result pollution —
each discovered tool is still a normal call whose raw result lands in context.

In [6]:
mcp = build_demo_server("tool-search")  # engine defaults to bm25
async with Client(mcp) as c:
    print("Visible tools:", [t.name for t in (await c.list_tools())])
    print()
    # Discover
    hits = (await c.call_tool("search_tools", {"query": "objects in a workspace"})).data
    print("search_tools('objects in a workspace') -> ranked:")
    for t in hits:
        print("   ", t["name"], "-", t["description"])
    print()
    # Inspect the full schema the model would receive for the top hit
    print("Full schema of top hit (this is what the model sees to build the call):")
    show(hits[0], 900)

Visible tools: ['search_tools', 'call_tool']

search_tools('objects in a workspace') -> ranked:
    list_objects - List geoscience objects in a workspace, optionally filtered by schema.
    list_users - List users in the current instance.
    count_points - Return the number of points in a pointset-like object.
    update_user_role - Change a user's role in the instance.
    get_workspace - Get details for a single workspace by id.

Full schema of top hit (this is what the model sees to build the call):
{
  "name": "list_objects",
  "description": "List geoscience objects in a workspace, optionally filtered by schema.",
  "inputSchema": {
    "additionalProperties": false,
    "properties": {
      "workspace_id": {
        "type": "string"
      },
      "schema": {
        "default": "",
        "type": "string"
      }
    },
    "required": [
      "workspace_id"
    ],
    "type": "object"
  },
  "outputSchema": {
    "properties": {
      "result": {
        "items": {
          

In [7]:
# Invoke a discovered tool through the call_tool proxy
async with Client(build_demo_server("tool-search")) as c:
    res = (
        await c.call_tool(
            "call_tool", {"name": "list_objects", "arguments": {"workspace_id": "ws-001", "schema": "pointset"}}
        )
    ).data
    print("call_tool(list_objects) ->")
    show(res)

call_tool(list_objects) ->
[
  {
    "id": "obj-b",
    "name": "topography",
    "schema": "pointset",
    "size": 98000
  },
  {
    "id": "obj-c",
    "name": "cu_grade",
    "schema": "pointset",
    "size": 45000
  }
]


### 3a. Regex engine vs BM25
`tool-search` supports two engines. BM25 ranks by natural-language relevance; regex does deterministic pattern matching. Choose via `MCP_SEARCH_ENGINE` / the `search_engine=` argument.

In [8]:
async with Client(build_demo_server("tool-search", search_engine="bm25")) as c:
    print(
        "BM25  'delete something':",
        [t["name"] for t in (await c.call_tool("search_tools", {"query": "delete something"})).data],
    )

async with Client(build_demo_server("tool-search", search_engine="regex")) as c:
    # regex uses a `pattern` argument, not `query`
    print(
        "Regex '^(delete|remove)':",
        [t["name"] for t in (await c.call_tool("search_tools", {"pattern": "^(delete|remove)"})).data],
    )

BM25  'delete something': ['delete_object']
Regex '^(delete|remove)': ['delete_object', 'remove_user']


## 4. Strategy `code-mode` — orchestrate in a sandbox
`list_tools()` returns `search`, `get_schema`, `execute`. The model discovers tools, then writes
**Python** that chains `await call_tool(name, params)` calls in a sandbox and returns only the final
answer. Intermediate results never touch the context window.

Solves: *too many tools upfront* **and** *round-trips* **and** *intermediate-result pollution*.
Cost: the model must write correct code (non-deterministic); needs a sandbox (`fastmcp[code-mode]`).

Inside the sandbox, `call_tool(name, params)` is the **only** function available — no client, no
token, no network, no imports. That is the key cloud-safety property.

In [8]:
mcp = build_demo_server("code-mode")
async with Client(mcp) as c:
    print("Visible tools:", [t.name for t in (await c.list_tools())])
    print()
    # Stage 1: search
    print("search('workspace objects') ->")
    print((await c.call_tool("search", {"query": "workspace objects points"})).data)

Visible tools: ['search', 'get_schema', 'execute']

search('workspace objects') ->
11 of 15 tools:

- list_objects: List geoscience objects in a workspace, optionally filtered by schema.
- create_pointset: Create a pointset object from a list of XYZ points.
- count_points: Return the number of points in a pointset-like object.
- get_workspace: Get details for a single workspace by id.
- create_workspace: Create a new workspace.
- workspace_health_check: Report health of the workspace and object services.
- upload_file: Upload a file into a workspace's file store.
- get_object: Get metadata for a single object.
- delete_object: Permanently delete an object. Destructive.
- get_object_versions: List the version history of an object.
- kriging_run: Run a kriging estimation job and return a job handle.


In [9]:
# Stage 2 + 3: get_schema then execute a multi-step workflow in ONE round-trip.
# Task: total points across all pointsets in ws-001 — 1 list + N counts, folded server-side.
workflow = """
objs = (await call_tool("list_objects", {"workspace_id": "ws-001", "schema": "pointset"}))["result"]
total = 0
for o in objs:
    r = await call_tool("count_points", {"workspace_id": "ws-001", "object_id": o["id"]})
    total += r["result"]
return {"pointsets": len(objs), "total_points": total}
"""
async with Client(build_demo_server("code-mode")) as c:
    result = (await c.call_tool("execute", {"code": workflow})).data
    print("execute(workflow) -> single final answer returned to the model:")
    show(result)

execute(workflow) -> single final answer returned to the model:
{
  "pointsets": 2,
  "total_points": 143000
}


Note the payoff: the loop above would be **1 + N** separate tool calls (and N raw result blobs in
context) under `none` or `tool-search`. Under `code-mode` the model sees **one** call and **one**
small result. `call_tool` inside the sandbox returns `{'result': <value>}` — hence the `["result"]`
unwrapping.

## 5. Guardrails: hiding destructive tools from the code/search path
Both transforms respect the FastMCP auth/visibility pipeline. Tools are tagged
(`read` / `write` / `admin` / `destructive`) in `demo_server.py`. A `Visibility` transform applied
*before* the strategy removes destructive tools from discovery entirely — so they can't be reached
via `search`/`execute`/`call_tool`, while remaining available as explicit, confirmable direct calls.

In [10]:
from demo_server import register_demo_tools
from fastmcp import FastMCP
from fastmcp.server.transforms import Visibility
from tool_strategy import ToolStrategy, apply_strategy


def build_guarded(strategy):
    mcp = FastMCP("Evo MCP PoC (guarded)")
    register_demo_tools(mcp)
    mcp.add_transform(Visibility(False, tags={"destructive"}))  # hide destructive tools first
    apply_strategy(mcp, ToolStrategy(strategy))
    return mcp


async with Client(build_guarded("tool-search")) as c:
    hits = [t["name"] for t in (await c.call_tool("search_tools", {"query": "delete remove object user"})).data]
    print("Guarded tool-search for 'delete remove object user' ->", hits)
    print("(delete_object / remove_user are absent — hidden from discovery)")

Guarded tool-search for 'delete remove object user' -> ['update_user_role', 'get_object', 'get_object_versions', 'count_points', 'workspace_health_check']
(delete_object / remove_user are absent — hidden from discovery)


## 6. Against the live Evo MCP server (discovery only)

Everything above used the mock **demo server**. This section targets the **real Evo MCP tool
catalog** — every tool registered by `src/mcp_tools.py` (~46 tools).

**Safety & scope.** We exercise **discovery/search only**: `list_tools`, `search_tools`, `search`,
`get_schema`. These merely *read the tool catalog* — they make **no Evo API calls and need no
credentials**. We deliberately do **not** call `call_tool` or run `execute` against Evo here; those
would hit a live tenant. The multi-step workflow is **presented** as the code the model *would* write,
not executed.

This is the convincing headline for the team: our **real** catalog collapsing from ~46 tools to 2–3
synthetic tools.

In [11]:
# Build the real Evo tool catalog under a chosen strategy — mirrors src/mcp_tools.py's
# "all" tool-filter registration, but parametrized so we can compare strategies side by side.
# Registration is just decorators: no auth, no network.
from fastmcp import FastMCP
from tool_strategy import ToolStrategy, apply_strategy

from evo_mcp.tools import (
    register_admin_tools,
    register_compute_tools,
    register_file_tools,
    register_filesystem_tools,
    register_general_tools,
    register_instance_users_admin_tools,
    register_object_builder_tools,
    register_object_staging_tools,
)


def build_live_server(strategy):
    """Fresh server with the *real* Evo tools registered and a strategy applied."""
    mcp = FastMCP("Evo MCP Server")
    register_general_tools(mcp)
    register_admin_tools(mcp)
    register_instance_users_admin_tools(mcp)
    register_filesystem_tools(mcp)
    register_object_builder_tools(mcp)
    register_file_tools(mcp)
    register_compute_tools(mcp)
    register_object_staging_tools(mcp)
    apply_strategy(mcp, ToolStrategy(strategy))
    return mcp


async def live_listing_size(strategy):
    mcp = build_live_server(strategy)
    async with Client(mcp) as c:
        tools = await c.list_tools()
        payload = [t.model_dump() for t in tools]
    return len(tools), len(json.dumps(payload, default=str))

### 6a. The real catalog collapse
Same measurement as §1a, but against the **actual Evo tool catalog**.

In [12]:
print(f"{'strategy':14}{'upfront tools':>14}{'schema chars':>14}{'~tokens':>10}")
print("-" * 52)
for strat in ["none", "tool-search", "code-mode"]:
    count, chars = await live_listing_size(strat)
    print(f"{strat:14}{count:>14}{chars:>14}{chars // 4:>10}")

strategy       upfront tools  schema chars   ~tokens
----------------------------------------------------
none                      42         54265     13566
tool-search                2          1316       329
code-mode                  3          2527       631


### 6b. Live discovery — `tool-search`
Real natural-language queries against the real tool names. Safe: `search_tools` only reads the
catalog. (We stop at discovery — no `call_tool`.)

In [13]:
async with Client(build_live_server("tool-search")) as c:
    for q in ["objects in a workspace", "kriging estimation", "download a file version", "manage users and roles"]:
        hits = (await c.call_tool("search_tools", {"query": q})).data
        print(f"search_tools({q!r}) -> {[t['name'] for t in hits]}")
    print()
    # The full schema the model would receive for one real tool:
    top = (await c.call_tool("search_tools", {"query": "kriging estimation"})).data[0]
    print("Full schema of top hit (what the model uses to build the call):")
    show(top, 1000)

search_tools('objects in a workspace') -> ['list_objects', 'preview_duplicate_scan', 'kriging_run', 'create_workspace_snapshot', 'find_duplicate_objects']
search_tools('kriging estimation') -> ['kriging_run', 'kriging_build_parameters']
search_tools('download a file version') -> ['download_file', 'workspace_copy_object', 'get_object', 'preview_csv_file', 'list_file_versions']
search_tools('manage users and roles') -> ['get_users_in_instance', 'add_users_to_instance', 'update_user_role_in_instance', 'list_roles_in_instance', 'remove_user_from_instance']

Full schema of top hit (what the model uses to build the call):
{
  "name": "kriging_run",
  "description": "Run one or more kriging compute tasks in parallel.",
  "inputSchema": {
    "additionalProperties": false,
    "properties": {
      "workspace_id": {
        "type": "string",
        "description": "Workspace UUID where all tasks should run."
      },
      "scenarios": {
        "items": {
          "description": "Parameters 

### 6c. Live discovery — `code-mode`
`search` then `get_schema` against the real catalog. Both are catalog reads — no Evo API calls.
We intentionally **do not call `execute`** here.

In [14]:
async with Client(build_live_server("code-mode")) as c:
    print("search('workspace objects and versions'):")
    print((await c.call_tool("search", {"query": "workspace objects and versions"})).data)
    print()
    print("get_schema(['list_objects', 'get_object_versions']):")
    print((await c.call_tool("get_schema", {"tools": ["list_objects", "get_object_versions"]})).data)

search('workspace objects and versions'):
34 of 42 tools:

- create_workspace_snapshot: Create a snapshot of all objects and their current versions in a workspace.
- list_file_versions: List all versions of a file in a workspace.
- preview_duplicate_scan: Preview workspaces and object counts before running a duplicate scan.

Lists all workspaces in the current instance with the number of objects
in each. Use this to decide which workspaces to include in a
`find_duplicate_objects` call. No objects are downloaded.

Returns:
    A dict with:
      - total_workspaces: number of workspaces in the instance
      - workspaces: list of {workspace_id, workspace_name, object_count}
        (object_count is -1 if listing failed for that workspace)
- list_objects: List objects in a workspace with optional filtering.
- workspace_duplicate_workspace: Duplicate entire workspace (all objects and data blobs).
- find_duplicate_objects: Find duplicate objects across Evo workspaces by comparing blob hashe

### 6d. The workflow the model *would* write (presented, not executed)
Under `code-mode`, after discovering the real tools above, the model would author a script like the
one below and call `execute(code=...)`. Running it hits live Evo and needs credentials + a real
`workspace_id`, so here we only **display** it — no execution.

> To actually run it, a reviewer *with* credentials and a selected instance would pass this string to
> the real server's `execute` tool for their own `workspace_id`.

In [15]:
# Presentation only — this cell prints the code, it does NOT execute it against Evo.
sample_workflow = """
# Summarise a workspace without flooding context: count objects per schema,
# and report total points across all pointsets — all folded server-side.
wid = "<your-workspace-id>"
objs = (await call_tool("list_objects", {"workspace_id": wid}))["result"]

by_schema = {}
total_points = 0
for o in objs:
    by_schema[o["schema"]] = by_schema.get(o["schema"], 0) + 1
    if o["schema"] == "pointset":
        r = await call_tool("count_points", {"workspace_id": wid, "object_id": o["id"]})
        total_points += r["result"]

# Only this small summary returns to the model — not the raw object list.
return {"object_count": len(objs), "by_schema": by_schema, "total_points": total_points}
"""
print(sample_workflow)
print("# (Not executed here — discovery-only section. Pass to execute() on the live server to run.)")


# Summarise a workspace without flooding context: count objects per schema,
# and report total points across all pointsets — all folded server-side.
wid = "<your-workspace-id>"
objs = (await call_tool("list_objects", {"workspace_id": wid}))["result"]

by_schema = {}
total_points = 0
for o in objs:
    by_schema[o["schema"]] = by_schema.get(o["schema"], 0) + 1
    if o["schema"] == "pointset":
        r = await call_tool("count_points", {"workspace_id": wid, "object_id": o["id"]})
        total_points += r["result"]

# Only this small summary returns to the model — not the raw object list.
return {"object_count": len(objs), "by_schema": by_schema, "total_points": total_points}

# (Not executed here — discovery-only section. Pass to execute() on the live server to run.)


## 7. Side-by-side summary
```
strategy       upfront tools  schema chars   ~tokens
----------------------------------------------------
none                      15          7257      1814
tool-search                2          1316       329
code-mode                  3          2527       631

Takeaways:
 - none:        deterministic, simplest; cost scales with catalog size.
 - tool-search: constant upfront cost; still 1 round-trip + raw result per call.
 - code-mode:   constant upfront cost; collapses multi-step workflows + hides intermediates.
 ```